# 1. Imports and Environment Setup
Install dependencies and load environment variables.

In [ ]:
# %pip install langchain langchain-openai openai python-dotenv langchain-community pypdf chromadb langchain-chroma langgraph langchain-huggingface rank-bm25 sentence-transformers pydantic

import os
import re
import json
import hashlib
import sqlite3
from time import perf_counter
from datetime import datetime
from typing import Optional, List, Dict, Any, Annotated

from dotenv import load_dotenv

try:
    from rank_bm25 import BM25Okapi
except ImportError:
    BM25Okapi = None

try:
    from sentence_transformers import CrossEncoder
except ImportError:
    CrossEncoder = None

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from pypdf import PdfReader
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage

from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver
from typing_extensions import TypedDict
from pydantic import BaseModel, Field

load_dotenv()
os.environ["OPENROUTER_API_KEY"] = ""
if "OPENROUTER_API_KEY" not in os.environ:
    raise ValueError("OPENROUTER_API_KEY not set")


# 2. Data Loading and Preprocessing
Load PDFs from the `mentaldocs` directory and perform Semantic Chunking using RecursiveCharacterTextSplitter as a robust alternative.

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

mentaldocs_path = "mentaldocs"
os.makedirs(mentaldocs_path, exist_ok=True)
all_docs = []

for file in os.listdir(mentaldocs_path):
    if file.endswith(".pdf"):
        path = os.path.join(mentaldocs_path, file)
        reader = PdfReader(path)
        for page_idx, page in enumerate(reader.pages):
            text = page.extract_text() or ""
            if text.strip():
                all_docs.append(
                    Document(
                        page_content=text,
                        metadata={"source": path, "page": page_idx},
                    )
                )

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks = text_splitter.split_documents(all_docs)
print(f"Loaded {len(all_docs)} pages, split into {len(chunks)} chunks.")


Ignoring wrong pointing object 491 0 (offset 0)
Ignoring wrong pointing object 501 0 (offset 0)
Ignoring wrong pointing object 560 0 (offset 0)
Ignoring wrong pointing object 568 0 (offset 0)
Ignoring wrong pointing object 601 0 (offset 0)
Ignoring wrong pointing object 609 0 (offset 0)
Ignoring wrong pointing object 668 0 (offset 0)
Ignoring wrong pointing object 676 0 (offset 0)
Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 27 0 (offset 0)
Ignoring wrong pointing object 32 0 (offset 0)
Ignoring wrong pointing object 34 0 (offset 0)
Ignoring wrong pointing object 43 0 (offset 0)
Ignoring wrong pointing object 56 0 (offset 0)
Multipl

Loaded 760 pages, split into 3483 chunks.


# 3. Embedding Generation
Initialize dense HuggingFace embeddings and BM25 tokenizers for sparse retrieval.

In [12]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

def tokenize_for_sparse(text: str) -> list:
    return re.findall(r"[a-z0-9]+", (text or "").lower())

chunk_texts = [d.page_content for d in chunks]
tokenized_chunks = [tokenize_for_sparse(t) for t in chunk_texts]

if BM25Okapi is not None and tokenized_chunks:
    bm25 = BM25Okapi(tokenized_chunks)
else:
    bm25 = None
    print("BM25Okapi not available or no chunks to process.")


# 4. Vector Database Initialization
Set up Chroma DB and internal memory tables.

In [13]:
if chunks:
    db = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory="doctor_kb",
        collection_metadata={"hnsw:space": "cosine"}
    )
    
    RETRIEVAL_K_DENSE = 6
    retriever = db.as_retriever(
        search_type="similarity",
        search_kwargs={"k": RETRIEVAL_K_DENSE}
    )
else:
    print("No chunks found. Ensure PDFs are in the 'mentaldocs' directory.")
    db = None
    retriever = None

conn = sqlite3.connect("compounder_memory.db")
cursor = conn.cursor()
cursor.execute("""
CREATE TABLE IF NOT EXISTS compounder_records (
    user_id INTEGER,
    timestamp TEXT,
    symptoms TEXT
)
""")
conn.commit()


# 5. Agent Definitions
Define State Dicts, Pydantic Schema Models, and LLMs.

In [14]:
llm_temperature_low = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0.1,
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
    openai_api_base="https://openrouter.ai/api/v1"
)

llm_temperature_med = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0.3,
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
    openai_api_base="https://openrouter.ai/api/v1"
)

import operator

class AgentState(TypedDict):
    user_id: int
    messages: Annotated[List[BaseMessage], operator.add]
    current_symptoms: dict
    followup_count: int
    session_active: bool
    active_query: str
    retrieved_docs: list
    final_context: str
    doctor_output: str
    final_reply: str

class SymptomExtraction(BaseModel):
    mood: str = Field(description="User's mood. Use 'unknown' if not mentioned.")
    anxiety: str = Field(description="Anxiety level. Use 'unknown' if not mentioned.")
    stress: str = Field(description="Stress level. Use 'unknown' if not mentioned.")
    sleep: str = Field(description="Sleep quality. Use 'unknown' if not mentioned.")
    energy: str = Field(description="Energy levels. Use 'unknown' if not mentioned.")
    appetite: str = Field(description="Appetite changes. Use 'unknown' if not mentioned.")
    duration: str = Field(description="Duration of symptoms. Use 'unknown' if not mentioned.")
    severity: str = Field(description="Severity from 1-10. Use 'unknown' if not mentioned.")

class PostCriticEval(BaseModel):
    is_grounded: bool = Field(description="True if the response is entirely grounded in the provided context and avoids prescribing medication.")
    feedback: str = Field(description="Feedback on what needs to change if not grounded.")
    refined_output: str = Field(description="A rewritten, safe, and grounded response if the original failed. If passed, leave empty.")


# 6. Retrieval Pipeline
Logic for Hybrid Retrieval (Dense + Sparse) with Reciprocal Rank Fusion.

In [15]:
RETRIEVAL_K_SPARSE = 8
FUSION_TOP_K = 8
FINAL_CONTEXT_TOP_K = 4
RRF_K = 60
cross_encoder = None

def get_reranker():
    global cross_encoder
    if CrossEncoder is None:
        return None
    if cross_encoder is None:
        cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
    return cross_encoder

def doc_key(doc) -> str:
    meta = getattr(doc, "metadata", {}) or {}
    source = str(meta.get("source", ""))
    page = str(meta.get("page", ""))
    body = getattr(doc, "page_content", "") or ""
    digest = hashlib.md5(body.encode("utf-8", errors="ignore")).hexdigest()
    return f"{source}|{page}|{digest}"

def retrieve_hybrid(query: str) -> list:
    if not retriever:
        return []
        
    dense_docs = retriever.invoke(query)
    
    sparse_docs = []
    if bm25 is not None:
        q_tokens = tokenize_for_sparse(query)
        scores = bm25.get_scores(q_tokens)
        ranked_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:RETRIEVAL_K_SPARSE]
        sparse_docs = [chunks[i] for i in ranked_idx]
    
    score_map = {}
    doc_map = {}
    for rank, d in enumerate(dense_docs, start=1):
        k = doc_key(d)
        score_map[k] = score_map.get(k, 0.0) + 1.0 / (RRF_K + rank)
        doc_map[k] = d
        
    for rank, d in enumerate(sparse_docs, start=1):
        k = doc_key(d)
        score_map[k] = score_map.get(k, 0.0) + 1.0 / (RRF_K + rank)
        doc_map[k] = d
        
    ranked = sorted(score_map.items(), key=lambda x: x[1], reverse=True)[:FUSION_TOP_K]
    fused_docs = [doc_map[k] for k, _ in ranked]
    
    reranker = get_reranker()
    if reranker is not None and fused_docs:
        pairs = [[query, d.page_content] for d in fused_docs]
        scores = reranker.predict(pairs)
        reranked = sorted(zip(fused_docs, scores), key=lambda x: x[1], reverse=True)
        final_docs = [d for d, _ in reranked[:FINAL_CONTEXT_TOP_K]]
    else:
        final_docs = fused_docs[:FINAL_CONTEXT_TOP_K]
        
    return final_docs

def docs_to_context(docs) -> str:
    lines = []
    for i, d in enumerate(docs, start=1):
        meta = getattr(d, "metadata", {}) or {}
        source = meta.get("source", "unknown")
        text = (getattr(d, "page_content", "") or "").strip()
        lines.append(f"[Source: {source}]\n{text}")
    return "\n\n---\n\n".join(lines)


# 7. Multimodal Processing
Placeholder hooks for Image/Audio inputs.

In [16]:
def process_multimodal_input(input_data: Any, input_type: str = "text") -> str:
    """
    Handles routing for multimodal input.
    """
    if input_type == "text":
        return input_data
    elif input_type == "audio":
        print("[System] Processing audio file...")
        # Audio transcription logic here (e.g., Whisper API)
        return "Audio transcription placeholder."
    elif input_type == "image":
        print("[System] Processing image file...")
        # Vision extraction logic here (e.g., extracting text from medical reports via GPT-4o Vision)
        return "Image extraction placeholder."
    return str(input_data)


# 8. Prompt Templates
Prompts with few-shot examples and strict system instructions.

In [17]:
compounder_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a calm, empathetic mental health intake assistant.
Your goal is to extract clinical symptoms into structured data.
Be very precise. If something is missing, mark it as 'unknown'.
Here are some few-shot examples:
User: "I haven't slept for 2 days and I'm very sad. It's a 8/10."
Output: mood='sad', sleep='has not slept for 2 days', severity='8', rest are 'unknown'.
"""),
    MessagesPlaceholder(variable_name="messages")
])
compounder_tool_llm = llm_temperature_low.with_structured_output(SymptomExtraction)

# followup_prompt = ChatPromptTemplate.from_messages([
#     ("system", """You are a calm, empathetic mental health assistant.
# Look at the missing fields in the symptoms below. 
# Ask ONE gentle follow-up question to clarify any missing core field (mood, anxiety, sleep).
# If all core fields and severity/duration are filled, output NO_FOLLOWUP_NEEDED.
# Symptoms so far: {symptoms}"""),
#     MessagesPlaceholder(variable_name="messages")
# ])
followup_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a calm, empathetic mental health assistant.
Look at the missing fields in the symptoms below. 
Decision rules:
- Check if you have sufficient information: AT LEAST ONE core field (mood, anxiety, or stress) MUST be known, AND 'duration' MUST be known, AND 'severity' MUST be known.
- If ALL of these conditions are met -> respond EXACTLY with: NO_FOLLOWUP_NEEDED
- Otherwise, ask ONE gentle, natural follow-up question to clarify ONE missing field.
Symptoms so far: {symptoms}"""),
    MessagesPlaceholder(variable_name="messages")
])

doctor_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a medical-support AI for mental health. You provide educational guidance based ONLY on the provided medical reference.
Do not invent conditions or provide medication dosages. Advise professional help if severe.
SYMPTOMS: {symptoms}
MEDICAL KNOWLEDGE:
{context}"""),
    ("human", "{query}")
])

post_critic_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a Post-Critic for a mental health chatbot.
Evaluate the following doctor output against the provided retrieved context.
Ensure the output:
1. Does NOT recommend specific medications.
2. Is strictly grounded in the context provided (no hallucinated medical claims).
If it fails, provide a rewritten safe output.
CONTEXT: {context}
DOCTOR OUTPUT: {doctor_output}
""")
])
post_critic_tool_llm = llm_temperature_low.with_structured_output(PostCriticEval)


# 9. Chatbot Execution Flow
LangGraph nodes, edges, and runner setup.

In [18]:
def get_user_input_node(state: AgentState) -> dict:
    user_input = interrupt("Waiting for user input...")
    processed_input = process_multimodal_input(user_input, "text")
    
    if processed_input.strip().lower() in ["exit", "quit", "bye"]:
        print("\nSession ended. Take care.")
        return {"session_active": False}
        
    return {"messages": [HumanMessage(content=processed_input)]}

def extract_symptoms_node(state: AgentState) -> dict:
    res: SymptomExtraction = compounder_tool_llm.invoke(compounder_prompt.format_prompt(messages=state["messages"]))
    new_symptoms = {k: v for k, v in res.dict().items() if v and v != "unknown"}
    current_symptoms = state.get("current_symptoms", {})
    current_symptoms.update(new_symptoms)
    return {"current_symptoms": current_symptoms}

def followup_node(state: AgentState) -> dict:
    symptoms = state.get("current_symptoms", {})
    
    core_filled = any(symptoms.get(f, "unknown") != "unknown" for f in ["mood", "anxiety", "stress"])
    duration_filled = symptoms.get("duration", "unknown") != "unknown"
    severity_filled = symptoms.get("severity", "unknown") != "unknown"
    
    if (core_filled and duration_filled and severity_filled) or state.get("followup_count", 0) >= 5:
        return {"active_query": f"Mental health guidelines for {json.dumps(symptoms)}"}
        
    unknown_fields = [f for f in ["mood", "anxiety", "stress", "sleep", "energy", "appetite", "duration", "severity"] if symptoms.get(f, "unknown") == "unknown"]
    
    res = llm_temperature_med.invoke(followup_prompt.format_prompt(
        unknown_fields=", ".join(unknown_fields),
        symptoms=json.dumps(symptoms),
        messages=state["messages"]
    ))
    content = res.content.strip()
    
    return {
        "messages": [AIMessage(content=content)],
        "followup_count": state.get("followup_count", 0) + 1
    }

# def retrieve_node(state: AgentState) -> dict:
#     query = state.get("active_query", "")
#     docs = retrieve_hybrid(query)
#     context = docs_to_context(docs)
#     return {"retrieved_docs": docs, "final_context": context}
def retrieve_node(state: AgentState) -> dict:
    query = state.get("active_query", "")
    
    if not query:
        symptoms = state.get("current_symptoms", {})
        query = f"Mental health guidelines for {json.dumps(symptoms)}"
        
    docs = retrieve_hybrid(query)
    context = docs_to_context(docs)
    return {"retrieved_docs": docs, "final_context": context}

def doctor_node(state: AgentState) -> dict:
    symptoms = state.get("current_symptoms", {})
    context = state.get("final_context", "")
    res = llm_temperature_med.invoke(doctor_prompt.format_prompt(
        symptoms=json.dumps(symptoms),
        context=context,
        query="Provide a gentle educational summary based on these symptoms and context."
    ))
    return {"doctor_output": res.content}

def critic_node(state: AgentState) -> dict:
    eval_res: PostCriticEval = post_critic_tool_llm.invoke(post_critic_prompt.format_prompt(
        context=state.get("final_context", ""),
        doctor_output=state.get("doctor_output", "")
    ))
    
    if not eval_res.is_grounded and eval_res.refined_output:
        final_reply = eval_res.refined_output
    else:
        final_reply = state.get("doctor_output", "")
        
    return {"final_reply": final_reply, "messages": [AIMessage(content=final_reply)]}


def route_after_input(state: AgentState) -> str:
    if not state.get("session_active", True):
        return "end"
    return "extract"

def route_after_followup(state: AgentState) -> str:
    if state.get("active_query"):
        return "retrieve"
    return "get_input"

builder = StateGraph(AgentState)
builder.add_node("get_input", get_user_input_node)
builder.add_node("extract", extract_symptoms_node)
builder.add_node("followup", followup_node)
builder.add_node("retrieve", retrieve_node)
builder.add_node("doctor", doctor_node)
builder.add_node("critic", critic_node)

builder.add_edge(START, "get_input")
builder.add_conditional_edges("get_input", route_after_input, {"extract": "extract", "end": END})
builder.add_edge("extract", "followup")
builder.add_conditional_edges("followup", route_after_followup, {"retrieve": "retrieve", "get_input": "get_input"})
builder.add_edge("retrieve", "doctor")
builder.add_edge("doctor", "critic")
builder.add_edge("critic", END)

graph = builder.compile(checkpointer=MemorySaver())

def run_chat():
    config = {"configurable": {"thread_id": "session_1"}}
    initial_state = {
        "user_id": 1,
        "messages": [],
        "current_symptoms": {},
        "followup_count": 0,
        "session_active": True
    }
    
    print("Agentic RAG Mental Health Assistant started. Type 'exit' to quit.")
    
    for _ in graph.stream(initial_state, config):
        pass
        
    printed_count = 0
    while True:
        state = graph.get_state(config)
        messages = state.values.get("messages", [])
        
        # Print any new AIMessages (either followups or final outputs)
        for msg in messages[printed_count:]:
            if isinstance(msg, AIMessage):
                prefix = "Assistant" if state.next else "Doctor"
                print(f"{prefix}: {msg.content}")
        
        printed_count = len(messages)
        
        if not state.next:
            break
            
        user_input = input("You: ")
        for event in graph.stream(Command(resume=user_input), config):
            for node, values in event.items():
                if node == "extract":
                    symptoms = values.get("current_symptoms", {})
                    full_symptoms = {k: symptoms.get(k, "unknown") for k in ["mood", "anxiety", "stress", "sleep", "energy", "appetite", "duration", "severity"]}
                    print(f"\n[System] Current symptom state:\n{json.dumps(full_symptoms, indent=2)}\n")
                elif node == "retrieve":
                    print("[System] Retrieving knowledge...")
                elif node == "doctor":
                    print("[System] Doctor is evaluating...")
                elif node == "critic":
                    print("[System] Reviewing safety...")

run_chat()

Agentic RAG Mental Health Assistant started. Type 'exit' to quit.

[System] Current symptom state:
{
  "mood": "depressed",
  "anxiety": "unknown",
  "stress": "unknown",
  "sleep": "unknown",
  "energy": "unknown",
  "appetite": "unknown",
  "duration": "unknown",
  "severity": "unknown"
}

Assistant: I'm sorry to hear that you're feeling this way. Can you share how long you've been experiencing these feelings?

[System] Current symptom state:
{
  "mood": "depressed",
  "anxiety": "unknown",
  "stress": "unknown",
  "sleep": "unknown",
  "energy": "unknown",
  "appetite": "unknown",
  "duration": "2-3 weeks",
  "severity": "high"
}

[System] Retrieving knowledge...
[System] Doctor is evaluating...
[System] Reviewing safety...
Doctor: Based on the symptoms you've described—persistent depressed mood for 2-3 weeks with high severity—it is important to recognize that these may indicate a condition known as depression. According to established guidelines, depression is characterized by a r